# 03 · Fixed-Cell Segmentation and Tracking
Segments cells across all fixed-cycle zarr acquisitions (Cy1–Cy10) for **BPP0050**.
Cy1 is the registration anchor. Cellpose handles segmentation; Trackastra handles tracking.
Labels are written back into each position's zarr store.

In [ ]:
import os
import re
import glob
import zarr
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from natsort import natsorted
from tqdm.auto import tqdm
from numcodecs import Blosc
from cellpose import models
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks
import seaborn as sns
import napari

## 1 · Initialise models

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

cellpose_model   = models.CellposeModel(gpu=(device == 'cuda'))
trackastra_model = Trackastra.from_pretrained('ctc', device=device)
compressor = Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)

BF_CH  = -1  # brightfield
DPC_CH = 0   # differential phase contrast

## 2 · Audit image dimensions across all fixed cycles
Confirms dimensionality consistency before running segmentation.

In [ ]:
root_dir  = '/mnt/DATA3/BPP0050/'
zarr_dirs = natsorted(glob.glob(os.path.join(root_dir, '*Fixed*/acquisition/zarr/*/')))

image_dims = {}
for zarr_dir in tqdm(zarr_dirs, desc='Auditing'):
    round_ID = zarr_dir.split('/')[-5]
    pos_ID   = zarr_dir.split('/')[-2]
    try:
        zg = zarr.open_group(zarr_dir)
        image_dims[(round_ID, pos_ID)] = zg.images.shape
    except Exception as e:
        print(f'Cannot open {zarr_dir}: {e}')

pd.DataFrame.from_dict(image_dims, orient='index', columns=['shape'])

## 3 · Segment and track Cy1 (registration anchor)

In [ ]:
include_path = Path(
    '/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Fixed_Cy1__2025-04-14T16_57_11-Measurement 1'
)
zarr_dir = include_path / 'acquisition/zarr'

for pos_zarr in tqdm(sorted(zarr_dir.glob('*.zarr')), desc='Segmenting Cy1'):
    zg = zarr.open_group(pos_zarr)
    bf  = zg.images[:, BF_CH,  ...].max(axis=1)
    dpc = zg.images[:, DPC_CH, ...].max(axis=1)
    seg_input = np.stack([bf, dpc], axis=1)  # (T, 2, Y, X)
    del bf, dpc

    masks, _, _       = cellpose_model.eval(seg_input, diameter=100, channel_axis=1)
    track_graph        = trackastra_model.track(masks, seg_input)
    masks_tracked, _  = graph_to_ctc(track_graph, masks, outdir=None)

    out = zarr.open_array(
        str(pos_zarr / 'labels' / 'segmentation'), mode='w',
        shape=masks_tracked.shape, chunks=(1, 1024, 1024),
        dtype=masks_tracked.dtype, compressor=compressor
    )
    out[:] = masks_tracked

## 4 · Segment and track all other fixed cycles (Cy2–Cy10)

In [ ]:
base_dirs   = natsorted(glob.glob(os.path.join(root_dir, '*Fixed*/')))
other_cycles = [b for b in base_dirs if str(include_path) not in b]

for base_dir in tqdm(other_cycles, desc='All fixed cycles'):
    pos_zarrs = natsorted(glob.glob(os.path.join(base_dir, 'acquisition/zarr/*.zarr')))
    for zarr_path_str in tqdm(pos_zarrs, desc=os.path.basename(base_dir), leave=False):
        zg = zarr.open_group(zarr_path_str)
        bf  = zg.images[:, BF_CH,  ...].max(axis=1)
        dpc = zg.images[:, DPC_CH, ...].max(axis=1)
        seg_input = np.stack([bf, dpc], axis=1)
        del bf, dpc

        masks, _, _      = cellpose_model.eval(seg_input, diameter=100, channel_axis=1)
        track_graph       = trackastra_model.track(masks, seg_input)
        masks_tracked, _ = graph_to_ctc(track_graph, masks, outdir=None)

        out = zarr.open_array(
            str(Path(zarr_path_str) / 'labels' / 'segmentation'), mode='w',
            shape=masks_tracked.shape, chunks=(1, 1024, 1024),
            dtype=masks_tracked.dtype, compressor=compressor
        )
        out[:] = masks_tracked

## 5 · QC visualisation in napari

In [ ]:
pos_zarr   = include_path / 'acquisition/zarr/(3, 4).zarr'
zg         = zarr.open_group(pos_zarr)
images     = zg.images[:]
segmentation = zg.labels.segmentation[:]

viewer = napari.Viewer(title='Fixed Cy1 — segmentation QC')
viewer.add_image(images, channel_axis=1)
viewer.add_labels(segmentation)
viewer.camera.zoom = 0.13
napari.run()

## 6 · Track length QC

In [ ]:
napari_tracks, _, _ = graph_to_napari_tracks(track_graph)
tracks = pd.DataFrame(napari_tracks, columns=['ID', 't', 'y', 'x'])
track_lengths = tracks.groupby('ID').size()

ax = sns.histplot(track_lengths, bins=30)
ax.figure.set_size_inches(6, 3)
sns.despine(offset=10)
ax.set(xlabel='Track length (frames)', ylabel='Count', title='Fixed-cell track length distribution')
print(f'Tracks spanning >= 23 frames: {(track_lengths >= 23).sum()}')